In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [2]:

df = pd.read_csv('T4Dataset.csv')
features = ['login_duration_min', 'data_accessed_MB', 'files_downloaded']


In [3]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])


In [ ]:

def flag_iqr(data, col):
    Q1, Q3 = data[col].quantile(0.25), data[col].quantile(0.75)
    IQR = Q3 - Q1
    return (data[col] < (Q1 - 1.5 * IQR)) | (data[col] > (Q3 + 1.5 * IQR))

df['iqr_anomaly'] = False 
for col in features:
    df['iqr_anomaly'] |= flag_iqr(df, col)

In [5]:

model = IsolationForest(contamination=0.05, random_state=42)
df['ml_anomaly'] = model.fit_predict(X_scaled)
df['ml_anomaly'] = df['ml_anomaly'].map({1: False, -1: True})

In [6]:

z_scores = (df[features] - df[features].mean()) / df[features].std()
df['risk_score'] = z_scores.abs().sum(axis=1)

In [7]:

suspects = df.sort_values(by='risk_score', ascending=False).head(5)
print("TOP 5 FORENSIC SUSPECTS:")
print(suspects[['user_id', 'remote_access', 'risk_score']])

TOP 5 FORENSIC SUSPECTS:
      user_id remote_access  risk_score
83   user_036           Yes   37.551257
79   user_033            No   27.211782
251  user_032           Yes   26.223458
314  user_045           Yes   14.868611
355  user_025           Yes    2.637406
